# GFP 후속 실험: minimizer 탐색과 enhancer 비교 · v0.1.6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/GFP_Benchmark_Followup_T4.ipynb?v=0.1.6)

T4 GPU 런타임에서 전체 실행을 선택합니다. 기본 설정은 아래 두 실험을 순서대로 실행합니다.

| 실험 | 구조 제공 | Recycles | 시드 |
| --- | --- | --- | --- |
| Minimizer | GFP와 결합체 각각의 구조 | 10 | 0–9 |
| Enhancer | GFP 구조만 | 10 | 0, 1, 2 |

총 13개 예측입니다. 실행 04의 소요시간을 단순 적용하면 추론에 약 40분이며, 설치·다운로드 시간은 별도입니다. 실험별 구조·설정·지표·로그를 구분해 저장하고 마지막 셀에서 하나의 ZIP으로 내려받습니다.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")

## 실험 선택

`all`은 두 실험을 실행합니다. `minimizer_seed_search`는 시드 확대만, `enhancer_target_only`는 결합체 템플릿 제거 비교만 실행합니다. 실행 조건과 시드 목록은 저장소의 `gfp_pipeline/benchmark_experiments.json`에서 읽습니다.


In [ ]:
import json

EXPERIMENT = "all"  # @param ["all", "minimizer_seed_search", "enhancer_target_only"]
settings = json.loads((checkout / "gfp_pipeline/benchmark_experiments.json").read_text(encoding="utf-8"))
experiments = [item for item in settings["experiments"]
               if EXPERIMENT == "all" or item["id"] == EXPERIMENT]
if not experiments:
    raise ValueError("실행할 실험을 선택하세요.")
for experiment in experiments:
    print(experiment["label"], "recycles:", experiment["recycles"], "seeds:", experiment["seeds"])
print("총 예측 수:", sum(len(item["seeds"]) for item in experiments))


In [ ]:
completed_runs = []
for experiment in experiments:
    print(f"실행: {experiment['label']}", flush=True)
    command = [sys.executable, str(checkout / "gfp_pipeline/run.py"),
               "--revision", revision, "--benchmark-only",
               "--benchmark-controls", experiment["control"],
               "--benchmark-template", experiment["template"],
               "--benchmark-recycles", str(experiment["recycles"]),
               "--benchmark-seeds", *map(str, experiment["seeds"])]
    result_zip = None
    with subprocess.Popen(command, stdout=subprocess.PIPE, text=True) as process:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
            if line.rstrip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"{experiment['id']}: 실행 실패. 위 로그를 확인하세요.")
    if result_zip is None or not result_zip.is_file():
        raise RuntimeError(f"{experiment['id']}: 결과 ZIP을 찾을 수 없습니다.")
    completed_runs.append({"experiment": experiment, "result_zip": result_zip,
                           "result_dir": result_zip.with_suffix("")})
    print(f"완료: {experiment['label']} — {result_zip.name}", flush=True)


## 실행 04와 비교

기존 회복 기준은 GFP RMSD ≤ 3 Å, GFP 접촉면 Jaccard ≥ 0.5, 충돌 없음입니다. 정확한 자세 판단에는 GFP 정렬 후 결합체 RMSD와 Cα 잔기 쌍 접촉 회복률도 함께 사용합니다. RMSD는 낮을수록, 회복률은 높을수록 기준 자세와 가깝습니다.

Minimizer는 탐색 확대 후 회복 자세가 생기는지와 그 자세의 ipTM 순위를 확인합니다. Enhancer는 GFP만 제공했을 때 실행 04의 `both` 조건과 비교합니다. 단독 대조군 실행은 이전 두 대조군 연속 실행과 난수 소비 순서가 달라질 수 있습니다.


In [ ]:
import pandas as pd
from IPython.display import display

baseline = json.loads((checkout / settings["baseline_summary"]).read_text(encoding="utf-8"))
comparison = []
for run in completed_runs:
    experiment = run["experiment"]
    current = json.loads((run["result_dir"] / "metrics/benchmark_summary.json").read_text(encoding="utf-8"))
    for label, report in (("실행 04", baseline), (experiment["id"], current)):
        for row in report["controls"]:
            if row["control"] == experiment["control"]:
                comparison.append({"run": label, "template": report["template_mode"], **row})
    poses = pd.read_csv(run["result_dir"] / "metrics/benchmark_poses.csv")
    print(experiment["label"])
    display(poses[["id", "seed", "iptm_rank", "iptm", "reference_pose_recovered",
                   "benchmark_contact_jaccard", "binder_ca_rmsd_after_target_alignment_angstrom",
                   "native_ca_contact_pair_recovery", "has_clash"]].round(3))
comparison_table = pd.DataFrame(comparison)
display(comparison_table[["run", "control", "template", "poses_tested", "poses_recovered",
                          "top_iptm", "top_iptm_binder_rmsd_angstrom",
                          "top_iptm_native_ca_contact_pair_recovery",
                          "best_binder_rmsd_angstrom", "best_recovered_iptm_rank"]].round(3))


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

bundle_dir = Path(mkdtemp(prefix="gfp_followup_", dir="/content"))
bundle = bundle_dir.with_suffix(".zip")
manifest = {"revision": revision, "baseline_summary": settings["baseline_summary"],
            "runs": [{"experiment": run["experiment"], "original_zip": run["result_zip"].name}
                     for run in completed_runs]}
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    archive.writestr("experiments.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    for run in completed_runs:
        for path in sorted(run["result_dir"].rglob("*")):
            if path.is_file():
                name = Path(run["experiment"]["id"]) / path.relative_to(run["result_dir"])
                archive.write(path, arcname=name.as_posix())
print(bundle)
files.download(str(bundle))
